# Association Rule Mining Pipeline

This notebook runs the full market basket analysis workflow using the Azure PostgreSQL retail warehouse as the source of truth.

It prepares basket-item data, applies a rare-item filter, splits baskets into train and test, runs Apriori, FP-Growth, and ECLAT with an adaptive support / confidence / lift grid, evaluates the rules as a recommender on the test baskets, records runtime and memory usage, and exports the final rule files plus an algorithm comparison table and a final accepted-metrics ranking.

## What You Will Produce

Expected outputs are written to `outputs/`:

- `basket_items_category.csv` (full dataset)
- `basket_items_category_train.csv` and `basket_items_category_test.csv` (when train/test split is enabled)
- `association_rules_apriori.csv`
- `association_rules_fpgrowth.csv`
- `association_rules_eclat.csv`
- `association_rules_comparison.csv` (every threshold combination per algorithm, with recommendation metrics and a final quality score)
- `final_accepted_association_metrics.csv` (one accepted row per algorithm, sorted by `final_quality_score`)

Generated CSVs are intentionally ignored by Git because they can be large.

## 1. Setup

Install dependencies first if needed:

```bash
pip install -r requirements.txt
```


In [1]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

required_modules = {
    "pandas": "pandas",
    "dotenv": "python-dotenv",
    "sqlalchemy": "sqlalchemy",
    "psycopg2": "psycopg2-binary",
    "psutil": "psutil",
    "mlxtend": "mlxtend",
}
missing_packages = [
    package_name
    for module_name, package_name in required_modules.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print("Installing missing packages into this notebook Python:")
    print(sys.executable)
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(PROJECT_ROOT / "requirements.txt"),
    ])
else:
    print("Python dependencies are available in this notebook kernel.")

import pandas as pd
from dotenv import load_dotenv

print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook Python: {sys.executable}")
print(f"Outputs folder: {OUTPUT_DIR}")

Installing missing packages into this notebook Python:
c:\Users\Mohamed\AppData\Local\Programs\Python\Python313\python.exe
Project root: c:\Users\Mohamed\Retail-Intelligence-DWH
Notebook Python: c:\Users\Mohamed\AppData\Local\Programs\Python\Python313\python.exe
Outputs folder: c:\Users\Mohamed\Retail-Intelligence-DWH\outputs


In [2]:
load_dotenv(PROJECT_ROOT / ".env")

required_env_vars = [
    "Azure_DB_HOST",
    "Azure_DB_USER",
    "Azure_DB_PASSWORD",
]

missing = [name for name in required_env_vars if not os.getenv(name)]
if missing:
    print("Missing database settings:", missing)
    print("Add them to a local .env file before running the extraction step.")
else:
    print("Database environment looks ready. Credentials were loaded without printing secrets.")

Database environment looks ready. Credentials were loaded without printing secrets.


## 2. Choose Mining Level and Parameters

Start with `product_category`. It is the safest level because the source has many product IDs.

**Threshold grid.** Association rule mining is sensitive to support and confidence. If thresholds are too strict, we get almost no rules. If they are too loose, we get many noisy rules. The mining script therefore performs an *adaptive threshold search*: it runs the grid from strictest to loosest and accepts the first run that produces a useful number of rules. If nothing falls inside the acceptable band, it falls back to the run with the rule count closest to the band.

**Rare-item filter.** Items appearing in very few baskets cannot produce stable rules and inflate runtime. `MIN_ITEM_FREQUENCY` is the minimum number of baskets an item must appear in. Keep it at `1` for category-level (already aggregated). Raise to around `50` for product-level mining.

**Train/test split.** When `TEST_SIZE > 0`, baskets are split deterministically using a hash of `RANDOM_STATE + basket_id`. Rules are mined on the train baskets only, then evaluated as a recommender on the test baskets.

In [ ]:
ITEM_LEVEL = "product_category"  

# Adaptive threshold grid: (min_support, min_confidence, min_lift) tuples, strictest first.
THRESHOLD_GRID = [
    {"min_support": 0.010, "min_confidence": 0.60, "min_lift": 1.50},
    {"min_support": 0.005, "min_confidence": 0.50, "min_lift": 1.30},
    {"min_support": 0.003, "min_confidence": 0.40, "min_lift": 1.20},
    {"min_support": 0.001, "min_confidence": 0.30, "min_lift": 1.00},
]

# Adaptive acceptance band: a run is accepted if its rule count falls in [MIN, MAX].
MIN_ACCEPTABLE_RULES = 20
MAX_ACCEPTABLE_RULES = 5000

MAX_ITEMSET_SIZE = 3

# Minimum number of baskets an item must appear in to be kept.
MIN_ITEM_FREQUENCY = 1

# Train/test split for recommendation evaluation.
TEST_SIZE = 0.2
RANDOM_STATE = 42
TOP_K = 5


LIMIT_BASKETS = None

LEVEL_SUFFIX = {
    "product_category": "category",
    "product_type": "type",
    "product_id": "product_id",
}[ITEM_LEVEL]

BASKET_ITEMS_CSV = OUTPUT_DIR / f"basket_items_{LEVEL_SUFFIX}.csv"
TRAIN_ITEMS_CSV = OUTPUT_DIR / f"basket_items_{LEVEL_SUFFIX}_train.csv"
TEST_ITEMS_CSV = OUTPUT_DIR / f"basket_items_{LEVEL_SUFFIX}_test.csv"

print(f"Selected item level: {ITEM_LEVEL}")
print(f"Prepared data will be saved to: {BASKET_ITEMS_CSV}")
print(f"Train / test baskets will be saved to:")
print(f"  {TRAIN_ITEMS_CSV}")
print(f"  {TEST_ITEMS_CSV}")
print(f"Adaptive threshold grid contains {len(THRESHOLD_GRID)} combinations.")

Selected item level: product_category
Prepared data will be saved to: c:\Users\Mohamed\Retail-Intelligence-DWH\outputs\basket_items_category.csv
Train / test baskets will be saved to:
  c:\Users\Mohamed\Retail-Intelligence-DWH\outputs\basket_items_category_train.csv
  c:\Users\Mohamed\Retail-Intelligence-DWH\outputs\basket_items_category_test.csv
Adaptive threshold grid contains 4 combinations.


## 3. Prepare Basket-Item Data

This step connects to Azure PostgreSQL and reads distinct `basket_id` plus the selected item field from `retail_dw.stg_basket_items_for_mba`.

The script removes nulls, drops duplicate basket-item pairs, applies the rare-item filter, and (when `TEST_SIZE > 0`) writes a deterministic train/test split alongside the full CSV.

**Why filter rare products.** Items that appear in only a handful of baskets cannot meet any meaningful support threshold, so they only add noise and runtime to the matrix. Removing them does not mean rare products are bad; it just means they are not useful for stable association rules.

**Why split train/test.** We do not want to evaluate recommendations on the same baskets used to mine rules. The split is deterministic (hash of `RANDOM_STATE + basket_id`), so re-running gives the same partition.

In [4]:
def run_command(command):
    print("Running command:")
    print(" ".join(str(part) for part in command))
    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")


prepare_command = [
    sys.executable,
    str(SCRIPTS_DIR / "04_prepare_basket_matrix.py"),
    "--item-level",
    ITEM_LEVEL,
    "--output",
    str(BASKET_ITEMS_CSV),
    "--min-item-frequency",
    str(MIN_ITEM_FREQUENCY),
    "--test-size",
    str(TEST_SIZE),
    "--random-state",
    str(RANDOM_STATE),
]

if LIMIT_BASKETS is not None:
    prepare_command.extend(["--limit-baskets", str(LIMIT_BASKETS)])

run_command(prepare_command)

Running command:
c:\Users\Mohamed\AppData\Local\Programs\Python\Python313\python.exe c:\Users\Mohamed\Retail-Intelligence-DWH\scripts\04_prepare_basket_matrix.py --item-level product_category --output c:\Users\Mohamed\Retail-Intelligence-DWH\outputs\basket_items_category.csv --min-item-frequency 1 --test-size 0.2 --random-state 42


In [5]:
basket_items = pd.read_csv(BASKET_ITEMS_CSV)
display(basket_items.head(10))

basket_summary = pd.Series(
    {
        "basket_item_rows": len(basket_items),
        "unique_baskets": basket_items["basket_id"].nunique(),
        "unique_items": basket_items["item_name"].nunique(),
        "duplicate_basket_item_pairs": basket_items.duplicated(["basket_id", "item_name"]).sum(),
        "null_basket_id": basket_items["basket_id"].isna().sum(),
        "null_item_name": basket_items["item_name"].isna().sum(),
    }
).to_frame("value")

display(basket_summary)

if TEST_SIZE > 0 and TRAIN_ITEMS_CSV.exists() and TEST_ITEMS_CSV.exists():
    train_items = pd.read_csv(TRAIN_ITEMS_CSV)
    test_items = pd.read_csv(TEST_ITEMS_CSV)
    split_summary = pd.DataFrame(
        {
            "basket_item_rows": [len(train_items), len(test_items)],
            "unique_baskets": [train_items["basket_id"].nunique(), test_items["basket_id"].nunique()],
            "unique_items": [train_items["item_name"].nunique(), test_items["item_name"].nunique()],
        },
        index=["train", "test"],
    )
    display(split_summary)

,basket_id,item_name
0,31198437603,BAG SNACKS
1,31198437603,BAKED BREAD/BUNS/ROLLS
2,31198437603,BAKED SWEET GOODS
3,31198437603,BEEF
4,31198437603,CANDY - CHECKLANE
5,31198437603,CHEESE
6,31198437603,CHEESES
7,31198437603,CITRUS
8,31198437603,CONDIMENTS/SAUCES
9,31198437603,CRACKERS/MISC BKD FD


,value
basket_item_rows,1078691
unique_baskets,155691
unique_items,303
duplicate_basket_item_pairs,0
null_basket_id,0
null_item_name,0


,basket_item_rows,unique_baskets,unique_items
train,860725,124418,301
test,217966,31273,300


## 4. Run Apriori, FP-Growth, and ECLAT with Adaptive Thresholds

The mining script runs each algorithm across every `(min_support, min_confidence, min_lift)` combination in `THRESHOLD_GRID`, records runtime and memory, and applies adaptive acceptance per algorithm: the first run whose rule count is inside `[MIN_ACCEPTABLE_RULES, MAX_ACCEPTABLE_RULES]` is accepted. If no run falls inside the band, the run with the rule count closest to the band is kept.

When `TEST_SIZE > 0` and a test CSV is passed via `--test-input`, every run is also evaluated as a recommender using `precision@5`, `recall@5`, `hit_rate@5`, and `coverage`. The recommendation score for ranking candidate consequents is `0.45*lift + 0.35*confidence + 0.20*support` (lift weighted highest because it measures association strength beyond random chance; support keeps the rule from being purely a rare-event artifact).

ECLAT is implemented with a vertical representation: `item_name -> set(basket_id)`.

In [ ]:
threshold_grid_arg = ",".join(
    f"{combo['min_support']}:{combo['min_confidence']}:{combo['min_lift']}"
    for combo in THRESHOLD_GRID
)

mining_input_csv = TRAIN_ITEMS_CSV if (TEST_SIZE > 0 and TRAIN_ITEMS_CSV.exists()) else BASKET_ITEMS_CSV

rules_command = [
    sys.executable,
    str(SCRIPTS_DIR / "05_association_rules.py"),
    "--item-level",
    ITEM_LEVEL,
    "--input",
    str(mining_input_csv),
    "--threshold-grid",
    threshold_grid_arg,
    "--max-itemset-size",
    str(MAX_ITEMSET_SIZE),
    "--min-acceptable-rules",
    str(MIN_ACCEPTABLE_RULES),
    "--max-acceptable-rules",
    str(MAX_ACCEPTABLE_RULES),
    "--top-k",
    str(TOP_K),
    "--random-state",
    str(RANDOM_STATE),
    "--output-dir",
    str(OUTPUT_DIR),
]

if TEST_SIZE > 0 and TEST_ITEMS_CSV.exists():
    rules_command.extend(["--test-input", str(TEST_ITEMS_CSV)])

run_command(rules_command)

Running command:
c:\Users\Mohamed\AppData\Local\Programs\Python\Python313\python.exe c:\Users\Mohamed\Retail-Intelligence-DWH\scripts\05_association_rules.py --item-level product_category --input c:\Users\Mohamed\Retail-Intelligence-DWH\outputs\basket_items_category_train.csv --threshold-grid 0.01:0.6:1.5,0.005:0.5:1.3,0.003:0.4:1.2,0.001:0.3:1.0 --max-itemset-size 3 --min-acceptable-rules 20 --max-acceptable-rules 5000 --top-k 5 --random-state 42 --output-dir c:\Users\Mohamed\Retail-Intelligence-DWH\outputs --test-input c:\Users\Mohamed\Retail-Intelligence-DWH\outputs\basket_items_category_test.csv


## 5. Compare Algorithm Performance

The full comparison table contains *every* threshold combination per algorithm, not just the accepted one. This is the right place to see how each algorithm scales as thresholds are relaxed: more itemsets, more rules, more runtime, more memory. The `accepted` flag marks the row chosen by adaptive search.

**Why we do not pick a winner on runtime alone.** A fast run that produces zero or noisy rules is useless. We track runtime and memory for engineering visibility, but the final ranking uses lift, confidence, and recommendation accuracy.

In [ ]:
comparison_path = OUTPUT_DIR / "association_rules_comparison.csv"
comparison = pd.read_csv(comparison_path)

display(
    comparison.sort_values(
        ["min_support", "min_confidence", "min_lift", "algorithm_name"],
        ascending=[True, True, True, True],
    )
)

performance_view = comparison[
    [
        "algorithm_name",
        "min_support",
        "min_confidence",
        "min_lift",
        "frequent_itemset_count",
        "rule_count",
        "avg_lift",
        "avg_confidence",
        "precision_at_5",
        "recall_at_5",
        "hit_rate_at_5",
        "coverage",
        "final_quality_score",
        "accepted",
        "runtime_seconds",
        "memory_delta_mb",
    ]
]

display(performance_view)

,algorithm_name,item_level,min_support,min_confidence,min_lift,frequent_itemset_count,rule_count,avg_support,avg_confidence,avg_lift,...,input_row_count,input_basket_count,input_item_count,precision_at_5,recall_at_5,hit_rate_at_5,coverage,final_quality_score,accepted,notes
3,Apriori,product_category,0.001,0.3,1.0,104312,98818,0.003107,0.453466,4.279522,...,860725,124418,301,0.112207,0.131721,0.382972,0.300000,0.535544,False,Frequent itemsets mined once per support thres...
11,ECLAT,product_category,0.001,0.3,1.0,104312,98818,0.003107,0.453466,4.279522,...,860725,124418,301,0.112207,0.131721,0.382972,0.300000,0.535544,False,ECLAT frequent itemsets mined once per support...
7,FP-Growth,product_category,0.001,0.3,1.0,104312,98818,0.003107,0.453466,4.279522,...,860725,124418,301,0.112207,0.131721,0.382972,0.300000,0.535544,False,Frequent itemsets mined once per support thres...
2,Apriori,product_category,0.003,0.4,1.2,24452,19727,0.006269,0.558931,3.658551,...,860725,124418,301,0.134964,0.133033,0.430722,0.130000,0.590004,False,Frequent itemsets mined once per support thres...
10,ECLAT,product_category,0.003,0.4,1.2,24452,19727,0.006269,0.558931,3.658551,...,860725,124418,301,0.134964,0.133033,0.430722,0.130000,0.590004,False,ECLAT frequent itemsets mined once per support...
6,FP-Growth,product_category,0.003,0.4,1.2,24452,19727,0.006269,0.558931,3.658551,...,860725,124418,301,0.134964,0.133033,0.430722,0.130000,0.590004,False,Frequent itemsets mined once per support thres...
1,Apriori,product_category,0.005,0.5,1.3,10007,5983,0.009133,0.629230,3.333662,...,860725,124418,301,0.173557,0.108920,0.395153,0.063333,0.619483,False,Frequent itemsets mined once per support thres...
9,ECLAT,product_category,0.005,0.5,1.3,10007,5983,0.009133,0.629230,3.333662,...,860725,124418,301,0.173557,0.108920,0.395153,0.063333,0.619483,False,ECLAT frequent itemsets mined once per support...
5,FP-Growth,product_category,0.005,0.5,1.3,10007,5983,0.009133,0.629230,3.333662,...,860725,124418,301,0.173557,0.108920,0.395153,0.063333,0.619483,False,Frequent itemsets mined once per support thres...
0,Apriori,product_category,0.010,0.6,1.5,2381,969,0.014987,0.669811,3.120423,...,860725,124418,301,0.165680,0.048170,0.245517,0.026667,0.380741,True,Frequent itemsets mined once per support thres...


,algorithm_name,min_support,min_confidence,min_lift,frequent_itemset_count,rule_count,avg_lift,avg_confidence,precision_at_5,recall_at_5,hit_rate_at_5,coverage,final_quality_score,accepted,runtime_seconds,memory_delta_mb
0,Apriori,0.010,0.6,1.5,2381,969,3.120423,0.669811,0.165680,0.048170,0.245517,0.026667,0.380741,True,13.2841,1.6133
1,Apriori,0.005,0.5,1.3,10007,5983,3.333662,0.629230,0.173557,0.108920,0.395153,0.063333,0.619483,False,44.5496,11.2695
2,Apriori,0.003,0.4,1.2,24452,19727,3.658551,0.558931,0.134964,0.133033,0.430722,0.130000,0.590004,False,35.9778,26.0312
3,Apriori,0.001,0.3,1.0,104312,98818,4.279522,0.453466,0.112207,0.131721,0.382972,0.300000,0.535544,False,41.8784,117.1054
4,FP-Growth,0.010,0.6,1.5,2381,969,3.120423,0.669811,0.165680,0.048170,0.245517,0.026667,0.380741,True,7.1825,159.3906
5,FP-Growth,0.005,0.5,1.3,10007,5983,3.333662,0.629230,0.173557,0.108920,0.395153,0.063333,0.619483,False,17.0502,26.1836
6,FP-Growth,0.003,0.4,1.2,24452,19727,3.658551,0.558931,0.134964,0.133033,0.430722,0.130000,0.590004,False,21.7894,31.3281
7,FP-Growth,0.001,0.3,1.0,104312,98818,4.279522,0.453466,0.112207,0.131721,0.382972,0.300000,0.535544,False,54.5183,36.3789
8,ECLAT,0.010,0.6,1.5,2381,969,3.120423,0.669811,0.165680,0.048170,0.245517,0.026667,0.380741,True,3.4897,46.8125
9,ECLAT,0.005,0.5,1.3,10007,5983,3.333662,0.629230,0.173557,0.108920,0.395153,0.063333,0.619483,False,4.9961,6.0781


## 5b. Final Accepted Metrics and Best Algorithm

`final_accepted_association_metrics.csv` keeps only the accepted run per algorithm. Rows are sorted by `final_quality_score`, computed from min-max normalized metrics:

```
final_quality_score =
    0.35 * normalized_avg_lift
  + 0.25 * normalized_avg_confidence
  + 0.25 * normalized_hit_rate_at_5
  + 0.15 * normalized_precision_at_5
```

**Why these weights.** Lift gets the largest share because it measures association strength independent of how common the items are. Confidence shows reliability of the implication. Hit rate and precision verify that the rules actually produce useful recommendations on unseen baskets. Support and runtime are intentionally not in the final score because they reward different things (commonness, speed) than recommendation quality.

In [ ]:
final_accepted_path = OUTPUT_DIR / "final_accepted_association_metrics.csv"
final_accepted = pd.read_csv(final_accepted_path)

display_columns = [
    "algorithm_name",
    "min_support",
    "min_confidence",
    "min_lift",
    "rule_count",
    "avg_confidence",
    "avg_lift",
    "max_lift",
    "precision_at_5",
    "recall_at_5",
    "hit_rate_at_5",
    "coverage",
    "final_quality_score",
    "runtime_seconds",
    "memory_delta_mb",
]

display(final_accepted[display_columns])

if not final_accepted.empty:
    best = final_accepted.iloc[0]
    print("\nBest overall algorithm by final_quality_score")
    print("-" * 60)
    print(f"Algorithm:        {best['algorithm_name']}")
    print(f"min_support:      {best['min_support']}")
    print(f"min_confidence:   {best['min_confidence']}")
    print(f"min_lift:         {best['min_lift']}")
    print(f"rule_count:       {int(best['rule_count'])}")
    print(f"avg_confidence:   {best['avg_confidence']:.4f}")
    print(f"avg_lift:         {best['avg_lift']:.4f}")
    print(f"max_lift:         {best['max_lift']:.4f}")
    print(f"precision@5:      {best['precision_at_5']:.4f}")
    print(f"recall@5:         {best['recall_at_5']:.4f}")
    print(f"hit_rate@5:       {best['hit_rate_at_5']:.4f}")
    print(f"coverage:         {best['coverage']:.4f}")
    print(f"runtime_seconds:  {best['runtime_seconds']:.4f}")
    print(f"final_quality:    {best['final_quality_score']:.4f}")

,algorithm_name,min_support,min_confidence,min_lift,rule_count,avg_confidence,avg_lift,max_lift,precision_at_5,recall_at_5,hit_rate_at_5,coverage,final_quality_score,runtime_seconds,memory_delta_mb
0,Apriori,0.01,0.6,1.5,969,0.669811,3.120423,12.151841,0.16568,0.04817,0.245517,0.026667,0.380741,13.2841,1.6133
1,FP-Growth,0.01,0.6,1.5,969,0.669811,3.120423,12.151841,0.16568,0.04817,0.245517,0.026667,0.380741,7.1825,159.3906
2,ECLAT,0.01,0.6,1.5,969,0.669811,3.120423,12.151841,0.16568,0.04817,0.245517,0.026667,0.380741,3.4897,46.8125



Best overall algorithm by final_quality_score
------------------------------------------------------------
Algorithm:        Apriori
min_support:      0.01
min_confidence:   0.6
min_lift:         1.5
rule_count:       969
avg_confidence:   0.6698
avg_lift:         3.1204
max_lift:         12.1518
precision@5:      0.1657
recall@5:         0.0482
hit_rate@5:       0.2455
coverage:         0.0267
runtime_seconds:  13.2841
final_quality:    0.3807


## 6. Inspect Strong Rules

Quick interpretation guide:

- `support`: share of baskets containing both sides of the rule.
- `confidence`: probability of the consequent given the antecedent.
- `lift`: relationship strength compared with random co-occurrence. Values above 1 suggest a positive association. **Lift is the most important metric for cross-sell rules** because high confidence alone can be an artifact of a frequent consequent.

In [ ]:
rule_files = {
    "Apriori": OUTPUT_DIR / "association_rules_apriori.csv",
    "FP-Growth": OUTPUT_DIR / "association_rules_fpgrowth.csv",
    "ECLAT": OUTPUT_DIR / "association_rules_eclat.csv",
}

columns_to_show = [
    "algorithm_name",
    "min_support",
    "min_confidence",
    "min_lift",
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift",
]

for algorithm_name, path in rule_files.items():
    rules = pd.read_csv(path)
    print(f"\n{algorithm_name}: {len(rules):,} exported rules")
    if rules.empty:
        print("No rules met the selected thresholds.")
        continue
    available_columns = [column for column in columns_to_show if column in rules.columns]
    display(
        rules.sort_values(["lift", "confidence", "support"], ascending=False)
        .head(10)[available_columns]
    )


Apriori: 125,497 exported rules


,algorithm_name,min_support,min_confidence,min_lift,antecedents,consequents,support,confidence,lift
26679,NaN,NaN,NaN,NaN,BAKED BREAD/BUNS/ROLLS | FROZEN,REFRIGERATED,0.001358,0.498525,25.768796
26680,NaN,NaN,NaN,NaN,CHEESE | FROZEN,REFRIGERATED,0.001085,0.487365,25.191912
26681,NaN,NaN,NaN,NaN,DIAPERS & DISPOSABLES | TROPICAL FRUIT,BABY HBC,0.001085,0.330882,23.271747
26682,NaN,NaN,NaN,NaN,FROZEN | TROPICAL FRUIT,REFRIGERATED,0.001053,0.441077,22.799324
26683,NaN,NaN,NaN,NaN,AIR CARE | LAUNDRY DETERGENTS,LAUNDRY ADDITIVES,0.001013,0.353933,21.886473
26684,NaN,NaN,NaN,NaN,BLEACH | LAUNDRY DETERGENTS,LAUNDRY ADDITIVES,0.001624,0.345890,21.389162
26685,NaN,NaN,NaN,NaN,BATH TISSUES | CAT LITTER,CAT FOOD,0.001101,0.526923,21.250799
26686,NaN,NaN,NaN,NaN,FLUID MILK PRODUCTS | FROZEN,REFRIGERATED,0.001037,0.409524,21.168315
26687,NaN,NaN,NaN,NaN,CAT LITTER | SOUP,CAT FOOD,0.001214,0.520690,20.999405
26688,NaN,NaN,NaN,NaN,FROZEN,REFRIGERATED,0.002267,0.405172,20.943391



FP-Growth: 125,497 exported rules


,algorithm_name,min_support,min_confidence,min_lift,antecedents,consequents,support,confidence,lift
26679,NaN,NaN,NaN,NaN,BAKED BREAD/BUNS/ROLLS | FROZEN,REFRIGERATED,0.001358,0.498525,25.768796
26680,NaN,NaN,NaN,NaN,CHEESE | FROZEN,REFRIGERATED,0.001085,0.487365,25.191912
26681,NaN,NaN,NaN,NaN,DIAPERS & DISPOSABLES | TROPICAL FRUIT,BABY HBC,0.001085,0.330882,23.271747
26682,NaN,NaN,NaN,NaN,FROZEN | TROPICAL FRUIT,REFRIGERATED,0.001053,0.441077,22.799324
26683,NaN,NaN,NaN,NaN,AIR CARE | LAUNDRY DETERGENTS,LAUNDRY ADDITIVES,0.001013,0.353933,21.886473
26684,NaN,NaN,NaN,NaN,BLEACH | LAUNDRY DETERGENTS,LAUNDRY ADDITIVES,0.001624,0.345890,21.389162
26685,NaN,NaN,NaN,NaN,BATH TISSUES | CAT LITTER,CAT FOOD,0.001101,0.526923,21.250799
26686,NaN,NaN,NaN,NaN,FLUID MILK PRODUCTS | FROZEN,REFRIGERATED,0.001037,0.409524,21.168315
26687,NaN,NaN,NaN,NaN,CAT LITTER | SOUP,CAT FOOD,0.001214,0.520690,20.999405
26688,NaN,NaN,NaN,NaN,FROZEN,REFRIGERATED,0.002267,0.405172,20.943391



ECLAT: 125,497 exported rules


,algorithm_name,min_support,min_confidence,min_lift,antecedents,consequents,support,confidence,lift
26679,NaN,NaN,NaN,NaN,BAKED BREAD/BUNS/ROLLS | FROZEN,REFRIGERATED,0.001358,0.498525,25.768796
26680,NaN,NaN,NaN,NaN,CHEESE | FROZEN,REFRIGERATED,0.001085,0.487365,25.191912
26681,NaN,NaN,NaN,NaN,DIAPERS & DISPOSABLES | TROPICAL FRUIT,BABY HBC,0.001085,0.330882,23.271747
26682,NaN,NaN,NaN,NaN,FROZEN | TROPICAL FRUIT,REFRIGERATED,0.001053,0.441077,22.799324
26683,NaN,NaN,NaN,NaN,AIR CARE | LAUNDRY DETERGENTS,LAUNDRY ADDITIVES,0.001013,0.353933,21.886473
26684,NaN,NaN,NaN,NaN,BLEACH | LAUNDRY DETERGENTS,LAUNDRY ADDITIVES,0.001624,0.345890,21.389162
26685,NaN,NaN,NaN,NaN,BATH TISSUES | CAT LITTER,CAT FOOD,0.001101,0.526923,21.250799
26686,NaN,NaN,NaN,NaN,FLUID MILK PRODUCTS | FROZEN,REFRIGERATED,0.001037,0.409524,21.168315
26687,NaN,NaN,NaN,NaN,CAT LITTER | SOUP,CAT FOOD,0.001214,0.520690,20.999405
26688,NaN,NaN,NaN,NaN,FROZEN,REFRIGERATED,0.002267,0.405172,20.943391
